## 1. 폴더이름 정돈하기

In [1]:
import os
import glob
import shutil
import re
import pandas as pd
from tqdm.notebook import tqdm
import concurrent.futures
import numpy as np

# ===================================================================
# 📌 1. 환경 설정 및 경로 변수
# ===================================================================

# ⭐ 1-1. Google Drive의 '내 컴퓨터' 경로 (BASE_PATH는 수동 확인 필요)
BASE_PATH = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜"

# ⭐ 1-2. 현재 파일들이 혼재되어 있는 원본 경로
SOURCE_DIR = os.path.join(BASE_PATH, "연습용")

# ⭐ 1-3. 최종적으로 정리된 파일들이 저장될 목표 경로
TARGET_BASE_DIR = BASE_PATH

# Bed 번호 범위
MIN_BED_NUM = 0  # bed00 파일을 포함하기 위해 0으로 변경
MAX_BED_NUM = 95
ALL_BEDS = set(range(MIN_BED_NUM, MAX_BED_NUM + 1))

# ⭐ 1-4. 병렬 처리에 사용할 최대 스레드 수
MAX_WORKERS = 16


print(f"✅ 원본 경로: {SOURCE_DIR}")
print(f"✅ 대상 경로: {TARGET_BASE_DIR}")
print(f"✅ 병렬 작업 스레드 수: {MAX_WORKERS}\n")

✅ 원본 경로: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/연습용
✅ 대상 경로: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜
✅ 병렬 작업 스레드 수: 16



In [2]:
# ===================================================================
# 2. 파일 메타데이터 파싱 및 분류 함수 (⭐ JSON 우선 분류 로직 적용)
# ===================================================================

def parse_file_metadata(file_path):
    """
    파일 경로에서 필요한 메타데이터를 추출하고 대상 폴더를 결정합니다.
    JSON 파일 여부를 최우선으로 판단합니다.
    """
    filename = os.path.basename(file_path)

    # 패턴 정의
    rgb_thermal_pattern = r"(bed\d{1,3})_(\d{8})_(\d{6})_(cam|thermal)(\d+)\.(jpg|jpeg|png)"
    json_extension_pattern = r".*\.json$" # 최종 확장자가 .json 인지 확인
    # JSON 파일의 핵심 메타데이터 추출을 위한 패턴 (cam/thermal 부분은 필수가 아님)
    json_core_metadata_pattern = r"(bed\d{1,3})_(\d{8})_(\d{6})_.*"

    # ----------------------------------------------------------------------
    # ✨ 2-1. JSON 파일 우선 처리 (최종 확장자가 .json 인지 확인)
    # ----------------------------------------------------------------------
    if re.search(json_extension_pattern, filename, re.IGNORECASE):
        # JSON 파일에서 bed_num, date, time 정보 추출 시도
        json_metadata_match = re.match(json_core_metadata_pattern, filename, re.IGNORECASE)

        if json_metadata_match:
            bed_num_str = json_metadata_match.group(1)
            date_str = json_metadata_match.group(2)
            time_str = json_metadata_match.group(3)

            bed_num = int(re.findall(r'\d+', bed_num_str)[0])
            yy_date = date_str[2:]
            target_folder = "JSON"

            return {
                'file_path': file_path,
                'target_folder': target_folder,
                'bed_num': bed_num,
                'date': yy_date,
                'time': time_str,
                'is_json': True,
                'sort_key': f"{yy_date}_{bed_num:03d}_{target_folder}_{time_str}",
            }
        # JSON 파일이지만 필요한 메타데이터 패턴을 따르지 않는 경우 무시
        return None

    # ----------------------------------------------------------------------
    # 2-2. RGB/Thermal 이미지 파일 처리
    # ----------------------------------------------------------------------
    match = re.match(rgb_thermal_pattern, filename, re.IGNORECASE)

    if match:
        bed_num_str, date_str, time_str, camera_type, cam_num, _ = match.groups()
        bed_num = int(re.findall(r'\d+', bed_num_str)[0])
        yy_date = date_str[2:]

        # 파일 분류 로직
        target_base_folder = None
        if camera_type.lower() == 'cam':
            if cam_num in ['0', '1']:
                target_base_folder = "RGB_윗면"
            elif cam_num == '2':
                target_base_folder = "RGB_정면"

        elif camera_type.lower() == 'thermal':
            target_base_folder = "열화상"

        if target_base_folder: # 이미지 파일인 경우에만 '0. 원본' 추가
            target_folder = os.path.join(target_base_folder, "0. 원본")
        else:
            return None # 정의된 카메라 타입이 아닌 경우 무시

        return {
            'file_path': file_path,
            'target_folder': target_folder,
            'bed_num': bed_num,
            'date': yy_date,
            'time': time_str,
            'is_json': False,
            'sort_key': f"{yy_date}_{bed_num:03d}_{target_folder}_{time_str}",
        }

    # 양식에 맞지 않는 파일 무시
    return None

In [3]:
# ===================================================================
# 3. 병렬 처리 도우미 함수 (Copy/Delete) - (⭐ Delete 함수는 더 이상 사용되지 않음)
# ===================================================================

def move_file(row):
    """단일 파일을 이동하고 이동된 경로를 반환합니다."""
    original_path = row['original_path']
    target_sub_folder = row['target_folder']
    target_date_folder = row['date']
    final_filename = os.path.basename(original_path)

    # target_sub_folder는 이제 'RGB_윗면/0. 원본' 또는 'JSON'과 같은 형식으로 넘어옵니다.
    # target_date_folder는 'YYMMDD' 형식입니다.
    target_dir = os.path.join(TARGET_BASE_DIR, target_sub_folder, target_date_folder)

    os.makedirs(target_dir, exist_ok=True)
    final_target_path = os.path.join(target_dir, final_filename)

    try:
        # shutil.move: 파일을 이동합니다.
        shutil.move(original_path, final_target_path)

        result = row.to_dict()
        result['moved_path'] = final_target_path
        return result
    except Exception as e:
        return {'error': f"이동 실패: {original_path} -> {final_target_path} ({e})"}


# ===================================================================
# 4. 메인 정리 로직 (⭐ 중복 삭제 로직 제거)
# ===================================================================

def run_data_reorganization_final():
    """
    중복 삭제 없이, JSON 우선 분리 및 병렬 처리를 사용하여 전체 파일을 이동합니다.
    (가장 나중에 이동된 파일이 덮어쓰여 최종적으로 남습니다.)
    """
    print("\n--- 1. 파일 검색 및 메타데이터 추출 시작 (연습용 폴더 전체 탐색) ---")
    all_file_data = []

    # ----------------------------------------------------------------------
    # 1-1. 파일 탐색 (SOURCE_DIR 전체를 os.walk로 재귀적으로 탐색)
    # ----------------------------------------------------------------------
    initial_count = 0
    if not os.path.exists(SOURCE_DIR):
        print(f"🚨 오류: 원본 경로 ({SOURCE_DIR})를 찾을 수 없습니다.")
        return None

    for root, _, files in os.walk(SOURCE_DIR):
        for filename in files:
            file_path = os.path.join(root, filename)
            metadata = parse_file_metadata(file_path)

            if metadata:
                metadata['original_path'] = file_path
                all_file_data.append(metadata)
                initial_count += 1

    print(f"✅ SOURCE_DIR 전체 탐색 완료: 총 {initial_count}개 유효 파일 발견.")

    if not all_file_data:
        print("🚨 경고: 분석할 유효한 양상추 이미지를 찾지 못했습니다.")
        return None

    df = pd.DataFrame(all_file_data)
    # Bed 1-95 범위 외 파일 필터링
    df_filtered = df[(df['bed_num'] >= MIN_BED_NUM) & (df['bed_num'] <= MAX_BED_NUM)].copy()

    print(f"총 유효 파일 수 (Bed {MIN_BED_NUM}-{MAX_BED_NUM} 기준): {len(df_filtered)}")

    # ⭐ 중복 삭제 로직 제거: df_filtered의 모든 파일을 이동 대상으로 사용
    df_move_targets = df_filtered.copy()

    # ===================================================================
    # 4-1. 파일 이동 (병렬 처리)
    # ===================================================================
    print(f"\n--- 2. 파일 이동 시작 (총 {len(df_move_targets)}개 파일, 병렬 처리) ---")
    print("⚠️ 중복된 시간대 파일이 있다면, 가장 나중에 이동되는 파일이 덮어쓰여 최종적으로 남게 됩니다.")

    moved_results = []

    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(move_file, row) for index, row in df_move_targets.iterrows()]

        # tqdm으로 진행 상황을 표시하며 결과 수집
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="파일 이동"):
            result = future.result()
            if isinstance(result, dict) and 'error' in result:
                print(f"🚨 이동 오류: {result['error']}")
            else:
                moved_results.append(result)

    df_final = pd.DataFrame(moved_results)
    print(f"\n✅ 이동 완료. 최종 이동된 파일 수: {len(df_final)}개")

    # ===================================================================
    # 5. 원본 '연습용' 폴더 정리 (수동 확인용 경고)
    # ===================================================================
    print("\n--- 3. 원본 '연습용' 폴더 정리 ---")

    # 원본 '연습용' 폴더 내의 모든 파일이 이동되었는지 확인
    remaining_files = []
    for root, _, files in os.walk(SOURCE_DIR):
        for file in files:
            # Check if the file still exists and is not a hidden file or system file (e.g., .DS_Store)
            if os.path.exists(os.path.join(root, file)) and not file.startswith('.'):
                remaining_files.append(os.path.join(root, file))

    if not remaining_files:
        try:
            print(f"✅ 모든 유효 파일이 성공적으로 이동되어 '{SOURCE_DIR}' 폴더를 삭제합니다.")
            shutil.rmtree(SOURCE_DIR)
        except Exception as e:
            print(f"🚨 오류: '{SOURCE_DIR}' 폴더 삭제 실패. 수동으로 삭제해주세요. ({e})")
    else:
        print(f"⚠️ '{SOURCE_DIR}' 폴더에 {len(remaining_files)}개의 파일이 남아있습니다. 수동으로 확인 및 삭제해주세요.")
        # 남아있는 파일 목록을 df_final에 추가하거나 별도 로깅 가능


    # ===================================================================
    # 6. 미수집 베드 현황 확인 및 알림
    # ===================================================================
    print("\n--- 4. 일별/카메라별 베드(Bed 1-95) 미수집 현황 확인 (전체 이동 파일 기준) ---")

    # 파일 이동 순서와 무관하게, 최종적으로 이동된 모든 유효 파일을 기준으로 누락 확인
    # target_folder는 이제 'RGB_윗면/0. 원본'과 같은 형태일 수 있으므로 이를 고려해야 합니다.
    # 이를 'RGB_윗면'으로 다시 매핑하여 그룹화하는 것이 더 적절할 수 있습니다.
    # 여기서는 'target_folder' 그대로 사용하므로 'RGB_윗면/0. 원본'으로 표시됩니다.
    collected_beds = df_final.groupby(['date', 'target_folder'])['bed_num'].apply(set).to_dict()
    missing_beds_found = False

    for (date, camera_type_path), beds_present in collected_beds.items():
        missing_beds = sorted(list(ALL_BEDS - beds_present))

        if missing_beds:
            missing_beds_found = True
            # display only the main folder name, not '0. 원본'
            display_camera_type = camera_type_path.split(os.sep)[0]
            print(f"❌ {date} - {display_camera_type} : 미수집 베드 ({len(missing_beds)}개). 예시: {missing_beds[:5]}... (총 {len(missing_beds)}개)")

    if not missing_beds_found:
        print("✅ 모든 날짜와 카메라 타입에서 Bed 1-95 데이터가 모두 수집되었습니다.")

    print("\n🎉 데이터 재정렬 및 이동 작업 완료! (df_final 변수에 최종 파일 목록 저장됨)")
    return df_final

In [4]:

# ===================================================================
# 🚀 스크립트 실행
# ===================================================================
df_final = run_data_reorganization_final()



--- 1. 파일 검색 및 메타데이터 추출 시작 (연습용 폴더 전체 탐색) ---
✅ SOURCE_DIR 전체 탐색 완료: 총 4039개 유효 파일 발견.
총 유효 파일 수 (Bed 0-95 기준): 4039

--- 2. 파일 이동 시작 (총 4039개 파일, 병렬 처리) ---
⚠️ 중복된 시간대 파일이 있다면, 가장 나중에 이동되는 파일이 덮어쓰여 최종적으로 남게 됩니다.


파일 이동:   0%|          | 0/4039 [00:00<?, ?it/s]


✅ 이동 완료. 최종 이동된 파일 수: 4039개

--- 3. 원본 '연습용' 폴더 정리 ---
⚠️ '/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/연습용' 폴더에 8개의 파일이 남아있습니다. 수동으로 확인 및 삭제해주세요.

--- 4. 일별/카메라별 베드(Bed 1-95) 미수집 현황 확인 (전체 이동 파일 기준) ---
❌ 251222 - JSON : 미수집 베드 (89개). 예시: [1, 2, 3, 4, 5]... (총 89개)
❌ 251222 - RGB_윗면 : 미수집 베드 (88개). 예시: [1, 2, 3, 4, 5]... (총 88개)
❌ 251222 - RGB_정면 : 미수집 베드 (90개). 예시: [0, 1, 2, 3, 4]... (총 90개)
❌ 251222 - 열화상 : 미수집 베드 (90개). 예시: [0, 1, 2, 3, 4]... (총 90개)
❌ 251224 - JSON : 미수집 베드 (37개). 예시: [14, 16, 18, 19, 21]... (총 37개)
❌ 251224 - RGB_윗면 : 미수집 베드 (54개). 예시: [8, 9, 10, 14, 15]... (총 54개)
❌ 251224 - RGB_정면 : 미수집 베드 (55개). 예시: [5, 8, 10, 14, 15]... (총 55개)
❌ 251224 - 열화상 : 미수집 베드 (50개). 예시: [14, 16, 18, 19, 21]... (총 50개)
❌ 251225 - JSON : 미수집 베드 (95개). 예시: [0, 1, 2, 3, 4]... (총 95개)
❌ 251225 - RGB_윗면 : 미수집 베드 (95개). 예시: [0, 1, 2, 3, 4]... (총 95개)
❌ 251225 - RGB_정면 : 미수집 베드 (95개). 예시: [0, 1, 2, 3, 4]... (총 95개)
❌ 251225 - 열화상 : 미수집 베

파일 진단

In [ ]:
import os
import glob
import pandas as pd
import re

# ===================================================================
# ᅕ 1. 환경 설정 및 경로 변수 (이 부분만 수정하세요)
# ===================================================================

# ★ 1-1. Google Drive의 '내 컴퓨터' 경로 (캡쳐를 기반으로 추정)
BASE_PATH = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜"

# ★ 1-2. 엑셀 파일 저장 경로 (원하는 경로로 수정)
OUTPUT_EXCEL_PATH = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추_분석_파일목록_251216.xlsx"

print(f"✅ 분석할 기준 경로: {BASE_PATH}")
print(f"✅ 결과 저장 경로: {OUTPUT_EXCEL_PATH}\n")


# ===================================================================
# 2. 파일 메타데이터 추출 함수
# ===================================================================

def extract_file_info(root_folder, sub_folder):
    """
    주어진 폴더 경로에서 파일 정보를 재귀적으로 추출합니다.
    """
    all_file_data = []

    # 해당 폴더 내의 모든 파일을 재귀적으로 검색
    search_path = os.path.join(root_folder, sub_folder, "**", "*")
    file_paths = glob.glob(search_path, recursive=True)

    for file_path in file_paths:
        if os.path.isfile(file_path):
            filename = os.path.basename(file_path)

            # 파일 이름에서 날짜, 시간, 베드, 카메라 정보 추출 시도
            match = re.search(r"(bed\d{1,3})_(\d{8})_(\d{6})_(cam|thermal)(\d+)", filename, re.IGNORECASE)

            bed_info, date_info, time_info, cam_type, cam_num = (None,) * 5
            if match:
                bed_info, date_str, time_info, cam_type, cam_num = match.groups()
                date_info = date_str[2:] # YYMMDD 형식

            all_file_data.append({
                '폴더명': sub_folder,
                '파일_경로_상대경로': file_path.replace(os.path.join(root_folder, sub_folder), '').lstrip('/'),
                '파일_이름': filename,
                '베드': bed_info,
                '날짜(YYMMDD)': date_info,
                '시간': time_info,
                '카메라_타입': cam_type + cam_num if cam_type and cam_num else None,
            })

    return pd.DataFrame(all_file_data)


# ===================================================================
# 3. 메인 실행 함수: 폴더 목록 확인 및 엑셀 저장
# ===================================================================

def generate_file_list_excel(base_path, output_path):

    # BASE_PATH 하위의 모든 폴더 목록을 가져옵니다.
    # 캡처 이미지 기준으로 'JSON', 'RGB_윗면', 'RGB_정면', '연습용', '열화상' 등을 포함합니다.
    sub_folders = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))]

    if not sub_folders:
        print("🚨 경고: 기준 경로에 하위 폴더가 없습니다. 경로를 확인해주세요.")
        return

    print(f"🔍 다음 {len(sub_folders)}개 폴더를 분석합니다: {sub_folders}")

    all_data_frames = {}

    # 폴더별로 파일을 추출하고 DataFrame을 생성
    for folder_name in sub_folders:
        print(f"--- 분석 시작: {folder_name} ---")
        df_result = extract_file_info(base_path, folder_name)

        if not df_result.empty:
            # 엑셀 시트 이름은 폴더명으로 사용합니다.
            sheet_name = folder_name.replace('/', '_').replace('\\', '_')
            all_data_frames[sheet_name] = df_result
            print(f"✅ {folder_name}: 총 {len(df_result)}개 파일 추출 완료.")
        else:
             print(f"⚠️ {folder_name}: 파일이 발견되지 않았습니다.")


    # 엑셀 파일로 저장
    if all_data_frames:
        print("\n--- 엑셀 파일 저장 시작 ---")
        try:
            # pandas.ExcelWriter를 사용하여 여러 시트에 저장
            with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
                for sheet_name, df in all_data_frames.items():
                    # 엑셀 시트 이름은 31자를 초과할 수 없으므로, 필요시 잘라줍니다.
                    df.to_excel(writer, sheet_name=sheet_name[:31], index=False)

            print(f"\nናህ 엑셀 저장 완료! {output_path}")

        except Exception as e:
            print(f"🚨 엑셀 저장 중 오류 발생: {e}")
            print("Google Drive 권한이나 경로를 확인해주세요.")
    else:
        print("🚨 저장할 데이터가 없어 엑셀 파일이 생성되지 않았습니다.")


# ===================================================================
# ናህ 스크립트 실행
# ===================================================================
# 먼저 Google Drive가 마운트되어 있는지 확인하고 실행하세요.
# from google.colab import drive
# drive.mount('/content/drive')

# xlsxwriter 모듈 설치
!pip install xlsxwriter

generate_file_list_excel(BASE_PATH, OUTPUT_EXCEL_PATH)


✅ 분석할 기준 경로: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜
✅ 결과 저장 경로: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추_분석_파일목록1.xlsx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 13.9 MB/s eta 0:00:00
🔍 다음 5개 폴더를 분석합니다: ['연습용', '열화상', 'JSON', 'RGB_윗면', 'RGB_정면']
--- 분석 시작: 연습용 ---
✅ 연습용: 총 26959개 파일 추출 완료.
--- 분석 시작: 열화상 ---
✅ 열화상: 총 4645개 파일 추출 완료.
--- 분석 시작: JSON ---
✅ JSON: 총 9281개 파일 추출 완료.
--- 분석 시작: RGB_윗면 ---
✅ RGB_윗면: 총 2593개 파일 추출 완료.
--- 분석 시작: RGB_정면 ---
✅ RGB_정면: 총 2354개 파일 추출 완료.

--- 엑셀 파일 저장 시작 ---

ናህ 엑셀 저장 완료! /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추_분석_파일목록1.xlsx
